# AbovePy Quickstart

A beginner-friendly walkthrough of **abovepy** — the Python library for accessing
KyFromAbove LiDAR, DEM, and orthoimagery data.

All data is publicly available from the KyFromAbove program. No credentials required.

**What you'll learn:**
- Explore available data products
- Search for tiles by county name or bounding box
- Download tiles to disk
- Stream raster data without downloading

In [ ]:
import abovepy

print(f"abovepy version: {abovepy.__version__}")

## Explore Available Products

KyFromAbove provides DEMs, orthoimagery, and LiDAR point clouds across multiple
collection phases. Use `abovepy.info()` to see what's available.

In [ ]:
# Show all available products as a table
products = abovepy.info()
products

## Search by County

The easiest way to find tiles is by county name. abovepy knows the bounding box
for all 120 Kentucky counties, so you don't need to look up coordinates.

In [ ]:
# Search for DEM Phase 3 tiles covering Franklin County
tiles = abovepy.search(county="Franklin", product="dem_phase3")

print(f"Found {len(tiles)} tiles")
tiles.head()

## Search by Bounding Box

For more precise control, pass a bounding box as `(xmin, ymin, xmax, ymax)` in
EPSG:4326 (longitude/latitude). This example covers an area near Lexington, KY.

In [ ]:
# Bounding box near Lexington, KY (EPSG:4326)
lexington_bbox = (-84.55, 38.00, -84.45, 38.07)

tiles_lex = abovepy.search(bbox=lexington_bbox, product="dem_phase3")

print(f"Found {len(tiles_lex)} tiles near Lexington")
tiles_lex.head()

## Download Tiles

Once you've found tiles, download them to a local directory.
Existing files are skipped by default (set `overwrite=True` to re-download).

In [ ]:
# Download the first 3 tiles to a local directory
paths = abovepy.download(tiles_lex.head(3), output_dir="./output")

print(f"Downloaded {len(paths)} files:")
for p in paths:
    print(f"  {p}")

## Stream Without Downloading

You can read raster data directly from the remote source without downloading
the full tile. This uses rasterio's `/vsicurl/` driver for efficient streaming.

Pass a `bbox` to read only a small spatial window.

In [ ]:
import numpy as np

# Read a small window from a remote tile
url = tiles_lex.iloc[0].asset_url
window_bbox = (-84.52, 38.02, -84.50, 38.04)

data, profile = abovepy.read(url, bbox=window_bbox)

print(f"Array shape: {data.shape}")
print(f"Data type:   {data.dtype}")
print(f"Min value:   {np.nanmin(data):.2f}")
print(f"Max value:   {np.nanmax(data):.2f}")
print(f"Mean value:  {np.nanmean(data):.2f}")
print(f"CRS:         {profile['crs']}")